# 12 | From the notebook-11 failure to a dispatch loss

**Purpose.** Notebook 11 shows that pointwise MSE and `optimize_day` are not the same objective. This notebook trains three day-level models (Transformer, LSTM, DLinear) with the **same block-sum plus window-ranking loss**, and asks whether the 59-day realized score can beat notebook 11's best score.

**Inputs:**

- Frozen notebook 11: `reports/holdout_rmse_v1/leaderboard.csv`, `reports/holdout_rmse_v2/leaderboard.csv`, `reports/holdout_rmse_v1/predictions.parquet`
- This trial: `reports/holdout_dispatch_loss_v1/` (run `experiments/holdout_dispatch_loss_v1/run.py` first if it is missing)

**Checks:**

- All three recipes have predicted score, realized score, and capture.
- Four figures go to `outputs/notebooks/figures/nb12_*.png`.
- The headline comparison uses dispatch score, not RMSE. The line to clear is notebook 11's best score, v2 Transformer (realized **6214**, capture **0.690**).
- Do not edit `reports/phase_b/final_metrics.json` or the v1/v2 RMSE columns, and do not treat these 59 days as a new champion selection.

**Score.** Official score `sum(price × power)`, not currency. `capture = realized / oracle`. The oracle enumerates 3,321 legal 8+8 windows on that day's realized `A`. Its 59-day mean is **9000.89** and is the same for every model. Realized score and capture rank models in the same order.

**Protocol.** 271 fit days / select `(lambda, tau)` on 30-day realized score / refit on 301 days / score the 59 days once. Early stopping does not look at RMSE. Selection also uses the 30-day realized score, so a gain versus notebook 11 mixes "change the loss" with "change the selection statistic."

These 59 days were already inspected in notebook 11. They are not an untouched holdout.

Stored outputs were removed from this public copy. The executed charts used Chinese labels and cannot be redrawn without the raw series. Numbers in the text are the recorded results.


## Read this first

- **Relative to notebook 11's best score, the dispatch loss clears the line.** The best new recipe is **`lstm_dispatch`**: realized **6301**, capture **0.700**, predicted score **6988**. Notebook 11's best, v2 Transformer, is 6214 / 0.690 / predicted 10527.
- **The change shows up in the LSTM.** In notebook 11, v1 LSTM is the weakest smoother (4554 / 0.506). The same head with the window-ranking loss reaches 6301 / 0.700.
- **The Transformer sits on notebook 11's line.** `transformer_dispatch` realized **6214**, capture **0.690**, almost the same as the v2 MSE Transformer. Predicted score is only 6740, so the optimism gap is smaller. It trained for 2 epochs (30-day realized score peaked at epoch 2). RMSE is 1.62, another sign that point error is not the dispatch score.
- **DLinear does not improve.** `dlinear_dispatch` realized **5538**, capture **0.615**, below the weekday-slot mean, with predicted score 8710 and a large gap.
- **These 59 days are not a new champion selection.** The window was already used in notebook 11. This notebook only asks whether the method scores higher than notebook 11's best. It does not refresh the 35-recipe board or the Phase B frozen file.


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'phase_b').is_dir():
            return candidate
    raise FileNotFoundError('start this notebook from the repository root')


def show_figure(path: Path) -> None:
    display(Image(filename=str(path)))


REPO_ROOT = find_repo_root()
os.environ.setdefault('MPLCONFIGDIR', str(REPO_ROOT / '.matplotlib_cache'))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES = REPO_ROOT / 'outputs' / 'notebooks' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
REPORT = REPO_ROOT / 'reports' / 'holdout_dispatch_loss_v1'
PHASE_B = REPO_ROOT / 'reports' / 'phase_b' / 'final_metrics.json'
MONEY_LINE_REALIZED = 6214.361782
MONEY_LINE_CAPTURE = 0.690417
print({'repo_root': str(REPO_ROOT), 'figures': str(FIGURES.relative_to(REPO_ROOT))})


## 1. What notebook 11 left open

Pointwise MSE averages 96 slots. `optimize_day` compares 3,321 legal 8+8 windows. v1 Transformer has RMSE 0.514 and realized score 5637, next to the weekday-slot mean 5644. v2 Transformer has RMSE 0.631 and realized score **6214**. v1 LSTM is the weakest smoother (4554 / 0.506). This notebook does not change the architecture. It changes the loss, and it runs all three compact heads.


## 2. The new loss, next to `F.mse_loss`

Previously: `L = mean((yhat - A)^2)` over 96 points.

Now: subtract the daily mean, take MSE on 89 eight-slot block sums, and train the predicted window scores to rank **that training day's true best window** first.

`s(tc, td) = S_td - S_tc`, **without** the factor 1000. `w*` comes only from the training day's realized price. Tree models do not have this differentiable softmax, and this notebook does not force one onto them.


In [ ]:
import inspect
from experiments.holdout_dispatch_loss_v1.loss import dispatch_loss

print(inspect.getsource(dispatch_loss))


In [ ]:
board_path = REPORT / 'leaderboard.csv'
# Published copy is read-only: it never launches training. Frozen experiment outputs
# under reports/ are not distributed with this repository -- see DATA.md for how to
# regenerate them from the source data before running this notebook.
assert board_path.exists(), (
    f'missing {board_path}. This notebook reads frozen experiment outputs only; '
    'it does not train. See DATA.md to obtain the data and regenerate reports/.'
)
print('loading frozen trial artifacts from', REPORT.relative_to(REPO_ROOT))

trial = pd.read_csv(REPORT / 'leaderboard.csv')
ok = trial[trial['status'] == 'ok'].copy()
assert len(ok) == 3, f'expected 3 ok recipes, got {len(ok)}: {trial["status"].tolist()}'
selection = json.loads((REPORT / 'selection.json').read_text(encoding='utf-8'))
manifest = json.loads((REPORT / 'manifest.json').read_text(encoding='utf-8'))
assert manifest.get('holdout_not_used_for_selection') is True
print(ok[['model', 'lam', 'tau', 'pred_profit_mean', 'realized_profit_mean', 'capture']].to_string(index=False))
print('phase_b frozen file still exists:', PHASE_B.exists())


In [ ]:
v1 = pd.read_csv(REPO_ROOT / 'reports' / 'holdout_rmse_v1' / 'leaderboard.csv')
v2 = pd.read_csv(REPO_ROOT / 'reports' / 'holdout_rmse_v2' / 'leaderboard.csv')


def _anchor(frame: pd.DataFrame, model: str, version: str) -> dict:
    row = frame.loc[frame['model'] == model].iloc[0]
    return {
        'source': version,
        'recipe': f'{version}/{model}',
        'pred_profit_mean': float(row['pred_profit_mean']),
        'realized_profit_mean': float(row['realized_profit_mean']),
        'capture': float(row['capture']),
        'profit_gap': float(row['profit_gap']),
        'oracle_profit_mean': float(row['oracle_profit_mean']),
        'RMSE': float(row['RMSE']),
    }


anchors = [
    _anchor(v2, 'transformer_contextual', 'v2'),
    _anchor(v1, 'transformer_contextual', 'v1'),
    _anchor(v1, 'lstm_contextual', 'v1'),
    _anchor(v1, 'climatology_weekday_slot', 'v1'),
]
new_rows = []
for _, row in ok.iterrows():
    new_rows.append({
        'source': 'nb12',
        'recipe': row['model'],
        'pred_profit_mean': float(row['pred_profit_mean']),
        'realized_profit_mean': float(row['realized_profit_mean']),
        'capture': float(row['capture']),
        'profit_gap': float(row['profit_gap']),
        'oracle_profit_mean': float(row['oracle_profit_mean']),
        'RMSE': float(row['RMSE']),
        'lam': float(row['lam']),
        'tau': float(row['tau']),
    })
compare = pd.DataFrame(anchors + new_rows)
best_new = ok.loc[ok['realized_profit_mean'].idxmax()]
beat = float(best_new['realized_profit_mean']) > MONEY_LINE_REALIZED
lstm_new = ok.loc[ok['model'] == 'lstm_dispatch'].iloc[0]
lstm_old = float(v1.loc[v1['model'] == 'lstm_contextual', 'realized_profit_mean'].iloc[0])

if beat:
    verdict = (
        f"Relative to notebook 11's best score, the dispatch loss clears the line. "
        f"The best new recipe is {best_new['model']}: realized {float(best_new['realized_profit_mean']):.0f}, "
        f"capture {float(best_new['capture']):.3f} (line 6214 / 0.690)."
    )
else:
    verdict = (
        f"The loss matches the 8+8 action, and this trial does not beat notebook 11's best score. "
        f"The best new recipe is {best_new['model']}: realized {float(best_new['realized_profit_mean']):.0f}, "
        f"capture {float(best_new['capture']):.3f}, against v2 Transformer 6214 / 0.690."
    )
verdict += (
    f" LSTM moves from v1 MSE at {lstm_old:.0f} to the dispatch loss at {float(lstm_new['realized_profit_mean']):.0f}."
    " These 59 days are not a new champion selection."
)
display(Markdown(f"**Result:** {verdict}"))
print(verdict)


## 3. Headline table: predicted score, realized score, capture

The line is drawn at v2 Transformer. RMSE stays in the table as an appendix column. It is not the decision.


In [ ]:
show = compare.copy()
show['pred_profit'] = show['pred_profit_mean'].round(0).astype(int)
show['realized'] = show['realized_profit_mean'].round(0).astype(int)
show['capture'] = show['capture'].round(3)
show['gap'] = show['profit_gap'].round(0).astype(int)
show['rmse'] = show['RMSE'].round(3)
show['beat_v2_tf'] = show['realized_profit_mean'] > MONEY_LINE_REALIZED
display(
    show[['source', 'recipe', 'pred_profit', 'realized', 'capture', 'gap', 'rmse', 'beat_v2_tf']]
    .sort_values('realized', ascending=False)
    .style.format({'capture': '{:.3f}', 'rmse': '{:.3f}'})
    .hide(axis='index')
)


In [ ]:
order = compare.sort_values('realized_profit_mean', ascending=False)
colors = ['#2a6f97' if src == 'nb12' else '#6c757d' for src in order['source']]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].barh(order['recipe'], order['realized_profit_mean'], color=colors)
axes[0].axvline(MONEY_LINE_REALIZED, color='#c44536', linestyle='--', linewidth=1, label='11 money line 6214')
axes[0].set_xlabel('realized mean (official score)')
axes[0].set_title('59-day realized profit')
axes[0].invert_yaxis()
axes[0].legend(loc='lower right', fontsize=8)
axes[1].barh(order['recipe'], order['capture'], color=colors)
axes[1].axvline(MONEY_LINE_CAPTURE, color='#c44536', linestyle='--', linewidth=1, label='11 money line 0.690')
axes[1].set_xlabel('capture = realized / 9000.89')
axes[1].set_title('59-day capture')
axes[1].invert_yaxis()
axes[1].legend(loc='lower right', fontsize=8)
fig.suptitle('Notebook 12 vs notebook 11 money anchors · blue = dispatch loss', fontsize=11)
fig.tight_layout()
p1 = FIGURES / 'nb12_fig1_realized_capture.png'
fig.savefig(p1, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p1.relative_to(REPO_ROOT))
show_figure(p1)


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
for src, sub in compare.groupby('source'):
    marker = 'o' if src == 'nb12' else 's'
    ax.scatter(
        sub['pred_profit_mean'],
        sub['realized_profit_mean'],
        s=70,
        marker=marker,
        label=src,
    )
    for _, row in sub.iterrows():
        ax.annotate(row['recipe'].replace('_contextual', '').replace('_weekday_slot', ''),
                    (row['pred_profit_mean'], row['realized_profit_mean']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')
lims = [
    min(compare['pred_profit_mean'].min(), compare['realized_profit_mean'].min()) - 200,
    max(compare['pred_profit_mean'].max(), compare['realized_profit_mean'].max()) + 200,
]
ax.plot(lims, lims, color='#888', linewidth=1, linestyle=':', label='pred = realized')
ax.axhline(MONEY_LINE_REALIZED, color='#c44536', linestyle='--', linewidth=1)
ax.set_xlabel('pred_profit mean (what optimize_day thought)')
ax.set_ylabel('realized mean (locked power × true A)')
ax.set_title('Honesty: predicted vs realized profit')
ax.legend(fontsize=8)
fig.tight_layout()
p2 = FIGURES / 'nb12_fig2_pred_vs_realized.png'
fig.savefig(p2, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p2.relative_to(REPO_ROOT))
show_figure(p2)


## 4. A negative day from notebook 11 (2025-12-12) and the new model

v1 Transformer realized −5343 that day. The question is whether the best dispatch-loss curve still places the window in the trough.


In [ ]:
from src.phase_b.dispatch import optimize_day
from src.phase_b.evaluation import score_day

v1_pred = pd.read_parquet(REPO_ROOT / 'reports' / 'holdout_rmse_v1' / 'predictions.parquet')
trial_pred = pd.read_parquet(REPORT / 'predictions.parquet')
best_name = str(best_new['model'])
time_col = 'times' if 'times' in v1_pred.columns else v1_pred.columns[0]
day = pd.Timestamp('2025-12-12')
v1_day = v1_pred.loc[pd.to_datetime(v1_pred[time_col]).dt.normalize() == day].copy()
tr_day = trial_pred.loc[pd.to_datetime(trial_pred['times']).dt.normalize() == day].copy()
actual = v1_day['A'].to_numpy(dtype=float) if 'A' in v1_day.columns else tr_day['A'].to_numpy(dtype=float)
old = v1_day['transformer_contextual'].to_numpy(dtype=float)
new = tr_day[best_name].to_numpy(dtype=float)
slots = np.arange(96)
old_dec = optimize_day(old)
new_dec = optimize_day(new)
ora = optimize_day(actual)

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(slots, actual, color='#111', linewidth=1.6, label='true A')
ax.plot(slots, old, color='#c44536', linewidth=1.4, label='v1 Transformer (MSE)')
ax.plot(slots, new, color='#2a6f97', linewidth=1.6, label=best_name)
if ora.tc is not None:
    ax.axvspan(ora.tc, ora.tc + 8, color='#6c757d', alpha=0.15, label='oracle charge')
    ax.axvspan(ora.td, ora.td + 8, color='#6c757d', alpha=0.28, label='oracle discharge')
if new_dec.tc is not None:
    ax.axvspan(new_dec.tc, new_dec.tc + 8, color='#2a6f97', alpha=0.12)
    ax.axvspan(new_dec.td, new_dec.td + 8, color='#2a6f97', alpha=0.22)
ax.set_xlabel('15-min slot')
ax.set_ylabel('price A')
ax.set_title(
    f'2025-12-12  v1 TF realized {score_day(actual, old_dec.power):.0f}  ·  '
    f'{best_name} realized {score_day(actual, new_dec.power):.0f}  ·  '
    f'oracle {score_day(actual, ora.power):.0f}'
)
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
p3 = FIGURES / 'nb12_fig3_dec12.png'
fig.savefig(p3, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p3.relative_to(REPO_ROOT))
show_figure(p3)


## 5. The 30-day selection grid

The 59-day window is not used here. Each model is trained on 271 days and `(lambda, tau)` is chosen on the 30-day realized score. The grid below is that selection, not the 59-day result.


In [ ]:
grid_rows = []
for family, payload in selection.items():
    for point in payload.get('grid', []):
        grid_rows.append({
            'family': family,
            'lam': point.get('lam'),
            'tau': point.get('tau'),
            'val30_realized': point.get('val30_realized'),
            'best_epoch': point.get('best_epoch'),
            'status': point.get('status'),
            'chosen': (
                point.get('lam') == payload.get('lam')
                and point.get('tau') == payload.get('tau')
                and point.get('status') == 'ok'
            ),
        })
grid = pd.DataFrame(grid_rows)
display(grid.sort_values(['family', 'val30_realized'], ascending=[True, False]).style.hide(axis='index'))

fig, ax = plt.subplots(figsize=(8.5, 4.0))
labels = [f"{r.family}\nλ={r.lam} τ={r.tau}" for r in grid.itertuples()]
vals = grid['val30_realized'].astype(float).to_numpy()
bar_colors = ['#2a6f97' if chosen else '#adb5bd' for chosen in grid['chosen']]
ax.bar(np.arange(len(grid)), vals, color=bar_colors)
ax.set_xticks(np.arange(len(grid)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('30-day realized (selection only)')
ax.set_title('λ, τ grid on the last 30 development days · blue = selected')
fig.tight_layout()
p4 = FIGURES / 'nb12_fig4_val30_grid.png'
fig.savefig(p4, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p4.relative_to(REPO_ROOT))
show_figure(p4)


## 6. These 59 days are not a new champion selection

The evaluation window was already used in notebook 11. This notebook only asks whether three day-level models, trained to rank 8+8 windows, score higher than notebook 11's best. It does not refresh the 35-recipe board, does not edit the Phase B frozen file, and does not retune `lambda` on these 59 days.
